In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import col, lag, lead, when, lit, current_timestamp

# 1. Leemos la tabla limpia de Silver
df_clean = spark.read.table("logistics_prod.silver.logistics_clean")

# 2. Definimos una ventana cronológica por cada camión (asset_id)
window_spec = Window.partitionBy("asset_id").orderBy("event_timestamp")

# 3. Detectamos cambios reales respecto a la lectura anterior (SCD Type 2 change detection)
df_changes = (
    df_clean
    .withColumn("prev_status", lag("shipment_status").over(window_spec))
    .withColumn("prev_inventory", lag("inventory_level").over(window_spec))
    .filter(
        col("prev_status").isNull() |                                  # Primer registro del camión
        (col("shipment_status") != col("prev_status")) |               # Cambió el estado del envío
        (col("inventory_level") != col("prev_inventory"))              # Cambió el nivel de inventario
    )
)

# 4. Calculamos los intervalos de validez (effective_from, effective_to, is_current)
scd2_window = Window.partitionBy("asset_id").orderBy("event_timestamp")

df_scd2 = (
    df_changes
    .withColumn("effective_from", col("event_timestamp"))
    .withColumn("effective_to", lead("event_timestamp").over(scd2_window))
    .withColumn("is_current", when(col("effective_to").isNull(), lit(True)).otherwise(lit(False)))
    .withColumn("updated_at", current_timestamp())
    .select(
        "asset_id",
        "shipment_status",
        "inventory_level",
        "traffic_status",
        "delay_reason",
        "effective_from",
        "effective_to",
        "is_current",
        "updated_at"
    )
)

# 5. Guardamos la tabla histórica SCD Type 2 en Silver
target_scd2_table = "logistics_prod.silver.asset_status_history"
df_scd2.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_scd2_table)

print(f"Tabla SCD Type 2 '{target_scd2_table}' generada con {spark.table(target_scd2_table).count()} registros históricos.")
